## 1. Install & Import Library

In [5]:
!pip install -q ultralytics onnx onnxsim

import json
import os
import random
import shutil
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

print("Ultralytics siap.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 3.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics siap.


## 2. Cek Struktur Dataset Mentah

**WAJIB dijalankan dulu.** Dataset SORDI bisa datang dalam beberapa variasi
struktur (instance-segmentation + label json, atau COCO json). Cell ini
men-scan folder input Kaggle dan menampilkan contoh isinya, supaya kita tahu
format apa yang harus dipakai di cell konversi (cell #3).


In [6]:
import kagglehub
from pathlib import Path

RAW_DATA_DIR = Path(kagglehub.dataset_download("mohamedessam3112002/sku-110k"))
print("Path to dataset files:", RAW_DATA_DIR)

Path to dataset files: /kaggle/input/datasets/mohamedessam3112002/sku-110k


In [7]:
import pandas as pd
import shutil
from pathlib import Path
from tqdm import tqdm

DATASET_ROOT = RAW_DATA_DIR / "SKU110K_fixed"
ANNOT_DIR = DATASET_ROOT / "annotations"
IMAGES_DIR = DATASET_ROOT / "images"
OUTPUT_DIR = Path("/kaggle/working/yolo_dataset")

COL_NAMES = ["image", "x1", "y1", "x2", "y2", "class", "image_width", "image_height"]

CLASS_LIST = ["object"]  # single-class dataset

In [8]:
# Daftar folder/file di root dataset
print("=== Isi root dataset ===")
for p in sorted(RAW_DATA_DIR.iterdir()):
    print(p.name, "-> folder" if p.is_dir() else "-> file")

=== Isi root dataset ===
SKU110K_fixed -> folder


In [9]:
# Kalau ada folder "images", intip 5 nama filenya
images_dir = RAW_DATA_DIR / "images"
if images_dir.exists():
    print("\n=== Contoh isi folder images/ ===")
    for f in list(images_dir.iterdir())[:5]:
        print(f.name)
else:
    print("Folder 'images' tidak ada langsung di root, perlu dicari di tempat lain.")

Folder 'images' tidak ada langsung di root, perlu dicari di tempat lain.


## 3. Konversi Anotasi ke Format YOLO


In [10]:
def convert_split(csv_name: str, split_out_name: str):
    df = pd.read_csv(ANNOT_DIR / csv_name, names=COL_NAMES, header=None)

    img_out = OUTPUT_DIR / "images" / split_out_name
    lbl_out = OUTPUT_DIR / "labels" / split_out_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    grouped = df.groupby("image")
    n_ok, n_skip = 0, 0

    for img_name, rows in tqdm(grouped, desc=f"Konversi {csv_name}"):
        src_img = IMAGES_DIR / img_name
        if not src_img.exists():
            n_skip += 1
            continue

        w = rows["image_width"].iloc[0]
        h = rows["image_height"].iloc[0]

        lines = []
        for _, r in rows.iterrows():
            x1, y1, x2, y2 = r["x1"], r["y1"], r["x2"], r["y2"]
            if x2 <= x1 or y2 <= y1:
                continue
            cx = ((x1 + x2) / 2) / w
            cy = ((y1 + y2) / 2) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")  # class id selalu 0

        if not lines:
            n_skip += 1
            continue

        shutil.copy(src_img, img_out / img_name)
        (lbl_out / (Path(img_name).stem + ".txt")).write_text("\n".join(lines))
        n_ok += 1

    print(f"{csv_name}: {n_ok} gambar berhasil, {n_skip} dilewati")

In [11]:
convert_split("annotations_train.csv", "train")
convert_split("annotations_val.csv", "val")
convert_split("annotations_test.csv", "test")

Konversi annotations_train.csv: 100%|██████████| 8219/8219 [03:23<00:00, 40.32it/s]


annotations_train.csv: 8219 gambar berhasil, 0 dilewati


Konversi annotations_val.csv: 100%|██████████| 588/588 [00:14<00:00, 41.11it/s]


annotations_val.csv: 588 gambar berhasil, 0 dilewati


Konversi annotations_test.csv: 100%|██████████| 2936/2936 [01:13<00:00, 40.00it/s]

annotations_test.csv: 2936 gambar berhasil, 0 dilewati


In [12]:
data_yaml_path = Path("/kaggle/working/data.yaml")
yaml_content = f"""path: {OUTPUT_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: object
"""
data_yaml_path.write_text(yaml_content)
print(yaml_content)

path: /kaggle/working/yolo_dataset
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: object



## 4. Training Model YOLOv8

In [13]:
MODEL_SIZE = "yolov8s.pt"   # 'n' lebih ringan (cocok untuk web), 'm' lebih akurat
EPOCHS = 40
IMG_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 20

model = YOLO(MODEL_SIZE)

results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    project="/kaggle/working/runs",
    name="sku-110k_yolov8",
    mosaic=1.0,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    val=True,
    plots=True,
    exist_ok=True,
)

BEST_WEIGHTS = Path("/kaggle/working/runs/sku-110k_yolov8/weights/best.pt")
print("Model terbaik:", BEST_WEIGHTS)


Ultralytics 8.4.165 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=5.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sku-110k_yolov8, nbs=64, nms=None, opset=None, optimize=False, op

Corrupt JPEG data: premature end of data segment


       1/40       5.9G      2.406      2.636      1.522       3802        640: 7% ╸─────────── 38/514 3.2it/s 22.7s<2:28

Corrupt JPEG data: 562 extraneous bytes before marker 0xd9


       1/40      7.36G      2.199      2.089      1.416       3820        640: 11% ━─────────── 57/514 2.7it/s 31.8s<2:49

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 89% ━━━━━━━━━━╸─ 17/19 1.7it/s 26.6s<1.2s

Corrupt JPEG data: 305 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 94% ━━━━━━━━━━━─ 18/19 1.4it/s 28.4s<0.7s

Corrupt JPEG data: 786 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.5s/it 28.8s
                   all        588      90968      0.907      0.837      0.889       0.55
Speed: 0.5ms preprocess, 11.0ms inference, 0.0ms loss, 13.3ms postprocess per image
Results saved to /kaggle/working/runs/sku-110k_yolov8
Model terbaik: /kaggle/working/runs/sku-110k_yolov8/weights/best.pt


## 5. Evaluasi di Test Set

In [14]:
eval_model = YOLO(str(BEST_WEIGHTS))
metrics = eval_model.val(data=str(data_yaml_path), split="test", plots=True)

print("mAP50:   ", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision (mean):", metrics.box.mp)
print("Recall (mean):   ", metrics.box.mr)


Ultralytics 8.4.165 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 180.2±89.9 MB/s, size: 1014.0 KB)
val: Scanning /kaggle/working/yolo_dataset/labels/test... 2935 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 2936/2936 192.9it/s 15.2s0.1s
val: /kaggle/working/yolo_dataset/images/test/test_1029.jpg: corrupt JPEG restored and saved
val: /kaggle/working/yolo_dataset/images/test/test_1035.jpg: corrupt JPEG restored and saved
val: /kaggle/working/yolo_dataset/images/test/test_1059.jpg: corrupt JPEG restored and saved
val: /kaggle/working/yolo_dataset/images/test/test_1086.jpg: corrupt JPEG restored and saved
val: /kaggle/working/yolo_dataset/images/test/test_1090.jpg: corrupt JPEG restored and saved
val: /kaggle/working/yolo_dataset/images/test/test_1102.jpg: corrupt JPEG restored and saved
val: /kaggle/working/yolo_dataset/images/test/t

Corrupt JPEG data: 935 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 17% ━━────────── 32/184 2.7it/s 24.5s<55.5s

Corrupt JPEG data: 70 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 19% ━━────────── 36/184 3.0it/s 25.8s<48.9s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 79% ━━━━━━━━━╸── 146/184 2.7it/s 1:05<14.2s

Corrupt JPEG data: 241 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 82% ━━━━━━━━━╸── 151/184 2.4it/s 1:07<13.6s

Corrupt JPEG data: 555 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 184/184 2.2it/s 1:240.5ss
                   all       2935     431419      0.913      0.866      0.928      0.577
Speed: 0.7ms preprocess, 6.4ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
mAP50:    0.927780932849842
mAP50-95: 0.5771049849625564
Precision (mean): 0.912594139694826
Recall (mean):    0.866445381404157


## 6. Tes Cepat Inferensi pada Beberapa Gambar

In [15]:
import glob

test_images = glob.glob(str(OUTPUT_DIR / "images" / "test" / "*"))[:5]
pred_results = eval_model.predict(
    source=test_images,
    conf=0.25,
    save=True,
    project="/kaggle/working/runs",
    name="inference_preview",
    exist_ok=True,
)

for r in pred_results:
    print(f"{r.path}: {len(r.boxes)} objek terdeteksi")
    for box in r.boxes:
        print(f"  - {r.names[int(box.cls[0])]}: {float(box.conf[0]):.2f}")



0: 640x640 173 objects, 12.2ms
1: 640x640 131 objects, 12.2ms
2: 640x640 119 objects, 12.2ms
3: 640x640 251 objects, 12.2ms
4: 640x640 120 objects, 12.2ms
Speed: 2.3ms preprocess, 12.2ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /kaggle/working/runs/inference_preview
/kaggle/working/yolo_dataset/images/test/test_505.jpg: 173 objek terdeteksi
  - object: 0.87
  - object: 0.86
  - object: 0.86
  - object: 0.86
  - object: 0.85
  - object: 0.85
  - object: 0.85
  - object: 0.85
  - object: 0.84
  - object: 0.84
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.83
  - object: 0.82
  - object: 0.82
  - object: 0.82
  - object: 0.82
  - object: 0.82
  - object: 0.82
  - object: 0.82
  - object: 0.81
  - object: 0.81
  - object: 0.81
  - object: 0.81
  - object: 0.81
  - object: 0.81
  - object: 0.81
  - object: 0.81
  - object: 0.81
  -

## 7. Export Model untuk Diserahkan ke Teman (Integrasi Web)

Tanya dulu teman kamu mau pakai library JS apa:
- **ONNX** → dipakai lewat `onnxruntime-web` (rekomendasi umum, lebih ringan)
- **TensorFlow.js** → dipakai lewat `tensorflow.js` (tfjs)


In [16]:
onnx_path = eval_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True, opset=12)
print("ONNX tersimpan di:", onnx_path)

# Kalau teman kamu butuh TensorFlow.js, uncomment baris di bawah:
# tfjs_path = eval_model.export(format="tfjs", imgsz=IMG_SIZE)
# print("TFJS tersimpan di:", tfjs_path)


Ultralytics 8.4.165 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/kaggle/working/runs/sku-110k_yolov8/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 299ms
Prepared 2 packages in 351ms
Installed 2 packages in 12ms
 + onnxruntime==1.30.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 3.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 6.2s

In [17]:
import os

# Memakai variabel 'onnx_path' langsung dari cell ekspor kamu
size_bytes = os.path.getsize(onnx_path)
size_mb = size_bytes / (1024 * 1024)

print(f"File ONNX Path: {onnx_path}")
print(f"Ukuran File ONNX: {size_mb:.2f} MB")

if size_mb > 30:
    print("⚠️ Warning: Ukuran cukup besar untuk PWA/Browser. Pertimbangkan YOLOv8n jika ingin loading time di browser lebih cepat.")
else:
    print("✅ Ukuran cukup ideal untuk edge/in-browser inference.")

File ONNX Path: /kaggle/working/runs/sku-110k_yolov8/weights/best.onnx
Ukuran File ONNX: 42.67 MB
⚠️ Warning: Ukuran cukup besar untuk PWA/Browser. Pertimbangkan YOLOv8n jika ingin loading time di browser lebih cepat.


In [18]:
import numpy as np
import onnxruntime as ort
from ultralytics import YOLO

# 1. Pilih 1 gambar sampel dari dataset/test kamu
sample_image_path = "/kaggle/input/datasets/mohamedessam3112002/sku-110k/SKU110K_fixed/images/test_1010.jpg"  # Ganti dengan path gambar test kamu

# 2. Inferensi dengan model PyTorch asli (eval_model)
pt_results = eval_model(sample_image_path)
pt_boxes = pt_results[0].boxes.data.cpu().numpy()

# 3. Inferensi dengan model ONNX yang baru diekspor via Ultralytics runner
onnx_model = YOLO(onnx_path)
onnx_results = onnx_model(sample_image_path)
onnx_boxes = onnx_results[0].boxes.data.cpu().numpy()

# 4. Bandingkan hasil deteksi
print(f"Jumlah deteksi (PyTorch) : {len(pt_boxes)}")
print(f"Jumlah deteksi (ONNX)    : {len(onnx_boxes)}")

if len(pt_boxes) == len(onnx_boxes):
    max_diff = np.max(np.abs(pt_boxes - onnx_boxes))
    print(f"Perbedaan Presisi Maksimal (Max Absolute Diff): {max_diff:.6f}")

    if np.allclose(pt_boxes, onnx_boxes, atol=1e-3):
        print("✅ VERIFIKASI SUKSES: Hasil PyTorch dan ONNX identik!")
    else:
        print(
            "⚠️ ADA PERBEDAAN MINOR: Hasil mirip, terdapat sedikit pergeseran float precision."
        )
else:
    print(
        "❌ VERIFIKASI GAGAL: Jumlah objek yang terdeteksi berbeda antara PyTorch dan ONNX."
    )


image 1/1 /kaggle/input/datasets/mohamedessam3112002/sku-110k/SKU110K_fixed/images/test_1010.jpg: 640x480 288 objects, 47.9ms
Speed: 2.3ms preprocess, 47.9ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 480)
Loading /kaggle/working/runs/sku-110k_yolov8/weights/best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 211ms
Prepared 1 package in 1.25s
Installed 1 package in 90ms
 + onnxruntime-gpu==1.30.0

requirements: AutoUpdate success ✅ 1.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Using ONNX Runtime 1.30.0 with CPUExecutionProvider
WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...

image 1/1 /kaggle/input/datasets/mohamedessam3112002/sku-110k/SKU110K_fixed/images/test_1010.jpg: 640x640 216 objects, 160.8ms
Speed: 5.9ms preprocess, 160.8ms inference, 25.0

## 8. Kemas Hasil untuk Diunduh

Kaggle Notebook otomatis menyimpan semua isi `/kaggle/working/` sebagai
**Output** yang bisa diunduh dari panel kanan setelah notebook selesai
di-run ("Save Version" → "Save & Run All"). Cell ini opsional, hanya
merapikan file penting jadi satu folder.


In [19]:
import shutil as sh
import glob

deliverable_dir = Path("/kaggle/working/deliverable")
deliverable_dir.mkdir(exist_ok=True)

# 1. Copy model (PyTorch + ONNX) dan config kelas
sh.copy(BEST_WEIGHTS, deliverable_dir / "best.pt")
sh.copy(onnx_path, deliverable_dir / "best.onnx")
sh.copy(data_yaml_path, deliverable_dir / "data.yaml")

# 2. Ambil 3-5 gambar test + jalankan prediksi biar ada kotak deteksinya
test_images = glob.glob(str(OUTPUT_DIR / "images" / "test" / "*"))[:5]

eval_model.predict(
    source=test_images,
    conf=0.25,
    save=True,
    project=str(deliverable_dir),
    name="sample_predictions",
    exist_ok=True,
)

for img_path in Path(deliverable_dir / "sample_predictions").glob("*"):
    sh.copy(img_path, deliverable_dir / img_path.name)
sh.rmtree(deliverable_dir / "sample_predictions")

print("Siap diserahkan ke teman, isi folder /kaggle/working/deliverable:")
for f in sorted(deliverable_dir.iterdir()):
    print(f" - {f.name} ({f.stat().st_size / 1024:.1f} KB)")


0: 640x640 173 objects, 13.2ms
1: 640x640 131 objects, 13.2ms
2: 640x640 119 objects, 13.2ms
3: 640x640 251 objects, 13.2ms
4: 640x640 120 objects, 13.2ms
Speed: 2.3ms preprocess, 13.2ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /kaggle/working/deliverable/sample_predictions
Siap diserahkan ke teman, isi folder /kaggle/working/deliverable:
 - best.onnx (43697.7 KB)
 - best.pt (21966.1 KB)
 - data.yaml (0.1 KB)
 - test_1577.jpg (2125.8 KB)
 - test_208.jpg (2880.9 KB)
 - test_2418.jpg (3027.6 KB)
 - test_505.jpg (3345.7 KB)
 - test_805.jpg (2354.0 KB)


In [21]:
zip_path = "/kaggle/working/model_deliverable"
shutil.make_archive(zip_path, "zip", deliverable_dir)
zip_full_path = zip_path + ".zip"
print(f"Zip siap: {zip_full_path} ({Path(zip_full_path).stat().st_size / 1024 / 1024:.2f} MB)")

Zip siap: /kaggle/working/model_deliverable.zip (68.74 MB)
